# SVD, conditioning, and least squares

1. The SVD in `numpy`
2. Condition number: the Hilbert matrix
3. Least squares: normal equations vs. `lstsq`
4. Truncated SVD: a toy unfolding problem

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## 1. The SVD in `numpy`

Any $M\times N$ matrix can be written $\mathbf{A}=\mathbf{U}\mathbf{\Sigma}\mathbf{V}^T$, with orthogonal $\mathbf{U}$, $\mathbf{V}$ and non-negative singular values $\Sigma_{ii}$ in decreasing order. `np.linalg.svd` returns `U`, the singular values as a 1D array `S`, and `Vt` $=\mathbf{V}^T$.

Here is a $4\times3$ matrix whose third column is the sum of the first two, so its rank is 2:

In [ ]:
A = np.array([[1., 2., 3.],
              [4., 5., 9.],
              [7., 8., 15.],
              [1., 0., 1.]])
U, S, Vt = np.linalg.svd(A, full_matrices=False)
print("singular values:", S)
print("rank (numpy):   ", np.linalg.matrix_rank(A))
print("max |U S Vt - A| =", np.max(np.abs(U @ np.diag(S) @ Vt - A)))

The smallest singular value is zero up to rounding. The corresponding row of `Vt` spans the **nullspace**: it is a vector $\mathbf{x}$ with $\mathbf{A}\mathbf{x}=0$.

In [ ]:
x_null = Vt[-1]
print("nullspace vector:", x_null / x_null[0])    # expect (1, 1, -1)
print("A @ x_null =", A @ x_null)

## 2. Condition number: the Hilbert matrix

$\kappa=\Sigma_\text{max}/\Sigma_\text{min}$. When solving $\mathbf{A}\mathbf{x}=\mathbf{b}$, expect to lose about $\log_{10}\kappa$ significant digits.

The Hilbert matrix $H_{ij}=1/(i+j+1)$ is a famous ill-conditioned example. (It is the matrix of the normal equations for fitting a polynomial on $[0,1]$: see section 3.) Solve $\mathbf{H}\mathbf{x}=\mathbf{H}\cdot\mathbf{1}$, whose exact solution is $\mathbf{x}=(1,1,\dots,1)$:

In [ ]:
for N in [4, 6, 8, 10, 12, 14]:
    i = np.arange(N)
    H = 1.0 / (i[:, None] + i[None, :] + 1)
    x_true = np.ones(N)
    x = np.linalg.solve(H, H @ x_true)
    err = np.linalg.norm(x - x_true) / np.linalg.norm(x_true)
    print(f"N={N:2d}  cond={np.linalg.cond(H):.1e}  rel. error={err:.1e}")

The relative error tracks $\kappa\times10^{-16}$. By $N\approx13$, $\kappa$ exceeds $10^{16}$ and no digits are left.

## 3. Least squares: normal equations vs. `lstsq`

A $\chi^2$ fit solves the overdetermined system $\mathbf{A}\vec{a}\approx\vec{b}$, with design matrix $A_{ij}=f_j(x_i)/\sigma_i$. The textbook solution is the **normal equations**, $\vec{a}=(\mathbf{A}^T\mathbf{A})^{-1}\mathbf{A}^T\vec{b}$. But $\kappa(\mathbf{A}^T\mathbf{A})=\kappa(\mathbf{A})^2$!

`np.linalg.lstsq` works with $\mathbf{A}$ directly (via the SVD), so it only sees $\kappa(\mathbf{A})$.

Fit a degree-7 polynomial with monomials $1, x, \dots, x^7$ on $[0,1]$, to noise-free data generated with all coefficients equal to 1:

In [ ]:
x = np.linspace(0, 1, 100)
A = np.vander(x, 8, increasing=True)        # design matrix, A_ij = x_i^j
a_true = np.ones(8)
b = A @ a_true

a_normal = np.linalg.solve(A.T @ A, A.T @ b)
a_lstsq = np.linalg.lstsq(A, b, rcond=None)[0]

print(f"cond(A) = {np.linalg.cond(A):.1e},  cond(A^T A) = {np.linalg.cond(A.T @ A):.1e}")
print(f"error, normal equations: {np.linalg.norm(a_normal - a_true):.1e}")
print(f"error, lstsq (SVD):      {np.linalg.norm(a_lstsq - a_true):.1e}")

Monomials on $[0,1]$ all look alike at large power, so the columns of $\mathbf{A}$ are nearly parallel. Better-behaved bases help too: e.g. shift and scale $x$ to $[-1,1]$, or use orthogonal polynomials (`np.polynomial.legendre.legvander`).

**Try this:** redo the comparison with $x$ on $[-1,1]$, and then with the Legendre design matrix. What happens to $\kappa(\mathbf{A})$ and the errors?

## 4. Truncated SVD: toy unfolding

A detector smears a true spectrum $\mathbf{t}$ (30 bins) with a Gaussian resolution, described by the **response matrix** $R_{ij}$ = probability that an event in true bin $j$ is measured in bin $i$. We measure $\mathbf{d}$ = Poisson($\mathbf{R}\mathbf{t}$), and want to recover $\mathbf{t}$.

In [ ]:
n = 30
c = np.arange(n) + 0.5                                                   # bin centers
t = 1000 * np.exp(-c / 8) + 300 * np.exp(-0.5 * ((c - 18) / 2) ** 2)    # "true" spectrum
R = np.exp(-0.5 * ((c[:, None] - c[None, :]) / 1.5) ** 2)               # Gaussian smearing
R /= R.sum(axis=0)                                                       # columns sum to 1
d = rng.poisson(R @ t).astype(float)                                     # measured, with noise

Using the SVD, $\mathbf{R}^{-1}\mathbf{d}=\sum_i \mathbf{v}_i\,(\mathbf{u}_i\cdot\mathbf{d})/\Sigma_{ii}$. Noise in $\mathbf{d}$ gets divided by the small singular values and blows up. **Truncated SVD** keeps only the $k$ largest singular values: this is a simple form of *regularization*.

In [ ]:
U, S, Vt = np.linalg.svd(R)
naive = Vt.T @ ((U.T @ d) / S)
k = 12                                                                   # keep largest k singular values
trunc = Vt.T[:, :k] @ ((U.T @ d)[:k] / S[:k])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.semilogy(S, "o")
ax1.axvline(k - 0.5, color="gray", ls="--")
ax1.set(xlabel="index $i$", ylabel=r"$\Sigma_{ii}$",
        title=fr"singular values of $\mathbf{{R}}$ ($\kappa$ = {S[0]/S[-1]:.0e})")
ax2.step(c, t, where="mid", color="k", label=r"true $\mathbf{t}$")
ax2.plot(c, d, "s", ms=4, label=r"measured $\mathbf{d}$")
ax2.plot(c, naive, "-", alpha=0.6, label=r"$\mathbf{R}^{-1}\mathbf{d}$")
ax2.plot(c, trunc, "o-", label=f"truncated SVD ($k$={k})")
ax2.set(ylim=(-500, 1500), xlabel="bin")
ax2.legend(fontsize=9)
fig.tight_layout()
plt.show()

**Try this:**

- Plot $|\text{trunc}-\mathbf{t}|$ as a function of $k$. Small $k$ throws away real structure (bias); large $k$ lets the noise through (variance). Where is the best $k$?
- Make the resolution worse (width 1.5 → 3 bins). How do the singular values and the best $k$ change?
- Real analyses use more refined methods (Tikhonov regularization, iterative D'Agostini unfolding), but they share the same idea.